In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import fisher_exact
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
from patsy.contrasts import Treatment as PatsyTreatment

In [6]:
# Path configuration
figfolder = "docs/"

# --------------------------------
# Data loading and preprocessing
# --------------------------------
def load_data():
    """Load data from CSV files and ensure columns are properly typed."""
    try:
        dfw = pd.read_csv('csv-files/Kidney Stone Reviews - Reviews - WebMD.csv')
        dfa = pd.read_csv('csv-files/Kidney Stone Reviews - Reviews - Amazon.csv')
        dfr = pd.read_csv('csv-files/Kidney Stone Reviews - Reviews - Reddit.csv')
        
        # Define all columns that should be numeric
        numeric_columns = [
            'Helps overall with kidney stones',
            'Side effects mentioned',
            'Super high quality',
            'Says that has suffered from condition for a long time (>1 year)',
            'Someone who makes large amounts of stones (>10 total)',
            'Works as a prophylactic',
            'Asserts significant pain reduction',
            'Mentions breaking of stones',
            'Mentions shrinking of the stones',
            'Mentions softening of stones',
            'Mentions stone gravel/dust/dissolved',
            'Stone passed with no or almost no pain',
            'Helps overall with gallstones',
            'Overall Rating',
            'Stars'
        ]
        
        # Convert columns to numeric for all dataframes
        # For the 'Side effects mentioned' column specifically:
        for df in [dfw, dfa, dfr]:
            # For other numeric columns
            for col in numeric_columns:
                if col in df.columns and col != 'Side effects mentioned':
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    df[col] = df[col].fillna(0).astype(int)
            
            # For the side effects column, convert to binary based on whether it contains text
            if 'Side effects mentioned' in df.columns:
                df['Side effects mentioned'] = df['Side effects mentioned'].apply(
                    lambda x: 1 if isinstance(x, str) and x.strip() else 0
                )
                
        print(f"Successfully loaded WebMD: {len(dfw)} rows, Amazon: {len(dfa)} rows, Reddit: {len(dfr)} rows")
        return dfw, dfa, dfr
    except Exception as e:
        print(f"Error loading data: {e}")
        return None, None, None

In [32]:
# --------------------------------
# Table 1: Summary of Online Reviews
# --------------------------------
def create_review_summary_table(dfw, dfa, dfr):
    """Creates a summary table of reviews by treatment and platform."""
    # Get unique treatments across all platforms
    all_treatments = sorted(set(dfw['Medicine'].unique()) | 
                          set(dfa['Medicine'].unique()) | 
                          set(dfr['Medicine'].unique()))
    
    # Exclude non-kidney stone treatments
    excluded_treatments = ['Ashwagandha', 'Melatonin']
    all_treatments = [t for t in all_treatments if t not in excluded_treatments]
    
    # Create empty dataframe
    summary_df = pd.DataFrame(index=all_treatments)
    
    # Calculate counts for each platform
    for treatment in all_treatments:
        # WebMD
        webmd_count = len(dfw[dfw['Medicine'] == treatment])
        webmd_quality = len(dfw[(dfw['Medicine'] == treatment) & 
                                  (dfw['Super high quality'] == 1)])
        
        # Amazon
        amazon_count = len(dfa[dfa['Medicine'] == treatment])
        amazon_quality = len(dfa[(dfa['Medicine'] == treatment) & 
                                   (dfa['Super high quality'] == 1)])
        
        # Reddit
        reddit_count = len(dfr[dfr['Medicine'] == treatment])
        reddit_quality = len(dfr[(dfr['Medicine'] == treatment) & 
                                   (dfr['Super high quality'] == 1)])
        
        # Add to dataframe
        summary_df.loc[treatment, 'WebMD Total'] = webmd_count
        summary_df.loc[treatment, 'WebMD HQ'] = webmd_quality
        summary_df.loc[treatment, 'Amazon Total'] = amazon_count
        summary_df.loc[treatment, 'Amazon HQ'] = amazon_quality
        summary_df.loc[treatment, 'Reddit Total'] = reddit_count
        summary_df.loc[treatment, 'Reddit HQ'] = reddit_quality
        summary_df.loc[treatment, 'Total Reviews'] = webmd_count + amazon_count + reddit_count
        summary_df.loc[treatment, 'Total HQ'] = webmd_quality + amazon_quality + reddit_quality
    
    # Fill NaNs with 0 and convert to integers
    summary_df = summary_df.fillna(0).astype(int)
    
    # Sort by total reviews descending
    summary_df = summary_df.sort_values('Total Reviews', ascending=False)
    
    # Save the table
    summary_df.to_html(figfolder + 'table1_review_summary.html')
    summary_df.to_csv(figfolder + 'table1_review_summary.csv')
    
    return summary_df

# --------------------------------
# Table 2: Effectiveness and Side Effects
# --------------------------------
def create_effectiveness_table(dfw, dfa, dfr):
    """Creates a table showing effectiveness and side effect rates across all platforms."""
    # Get unique treatments across all platforms
    all_treatments = sorted(set(dfw['Medicine'].unique()) | 
                          set(dfa['Medicine'].unique()) | 
                          set(dfr['Medicine'].unique()))
    
    # Exclude non-kidney stone treatments
    excluded_treatments = ['Ashwagandha', 'Melatonin']
    all_treatments = [t for t in all_treatments if t not in excluded_treatments]
    
    # Create empty dataframe
    results_df = pd.DataFrame(index=all_treatments)
    
    # Process each platform
    platforms = {
        'WebMD': dfw,
        'Amazon': dfa,
        'Reddit': dfr
    }
    
    for platform_name, df in platforms.items():
        for treatment in all_treatments:
            # Get subset for this treatment
            treatment_df = df[df['Medicine'] == treatment].copy()
            total_count = len(treatment_df)
            
            if total_count > 0:
                try:
                    # Calculate effectiveness percentage
                    effective_count = treatment_df['Helps overall with kidney stones'].sum()
                    effective_pct = round((effective_count / total_count) * 100, 1)
                    
                    # Calculate side effects percentage
                    side_effects_count = treatment_df['Side effects mentioned'].sum()
                    side_effects_pct = round((side_effects_count / total_count) * 100, 1)
                    
                    # Store in dataframe
                    results_df.loc[treatment, f'{platform_name} N'] = total_count
                    results_df.loc[treatment, f'{platform_name} Effective %'] = effective_pct
                    results_df.loc[treatment, f'{platform_name} Side Effects %'] = side_effects_pct
                
                except Exception as e:
                    print(f"Error processing {treatment} on {platform_name}: {e}")
                    # Add zeros for this treatment if there's an error
                    results_df.loc[treatment, f'{platform_name} N'] = total_count
                    results_df.loc[treatment, f'{platform_name} Effective %'] = 0
                    results_df.loc[treatment, f'{platform_name} Side Effects %'] = 0
    
    # Fill NaNs with 0
    results_df = results_df.fillna(0)
    
    # Only keep treatments that have at least one review
    results_df = results_df[results_df.filter(like='N').sum(axis=1) > 0]
    
    # Calculate average effectiveness and side effects across platforms
    results_df['Avg Effectiveness %'] = results_df.filter(like='Effective').mean(axis=1)
    results_df['Avg Side Effects %'] = results_df.filter(like='Side Effects').mean(axis=1)
    
    # Sort by average effectiveness
    results_df = results_df.sort_values('Avg Effectiveness %', ascending=False)
    
    # Save the table
    results_df.to_html(figfolder + 'table2_effectiveness_side_effects.html')
    results_df.to_csv(figfolder + 'table2_effectiveness_side_effects.csv')
    
    return results_df

# --------------------------------
# Helper Functions for Statistical Analysis
# --------------------------------
def calculate_odds_ratio(a, b, c, d):
    """Calculate odds ratio and 95% confidence interval for a 2x2 contingency table."""
    # Add 0.5 to each cell if any cell is 0 (Haldane-Anscombe correction)
    if a == 0 or b == 0 or c == 0 or d == 0:
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    
    # Calculate odds ratio
    odds_ratio = (a * d) / (b * c)
    
    # Calculate log standard error
    log_se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    
    # Calculate 95% confidence interval
    ci_lower = np.exp(np.log(odds_ratio) - 1.96 * log_se)
    ci_upper = np.exp(np.log(odds_ratio) + 1.96 * log_se)
    
    return odds_ratio, ci_lower, ci_upper

def pairwise_comparison(df, reference, comparison, outcome_col, high_quality=True):
    """
    Performs statistical comparison between reference treatment and comparison treatment.
    """
    # Create copies to avoid SettingWithCopyWarning
    df_copy = df.copy()
    
    # Filter for high-quality reviews if specified
    if high_quality:
        df_copy = df_copy[df_copy['Super high quality'] == 1]
    
    # Get subsets for each treatment
    ref_subset = df_copy[df_copy['Medicine'] == reference]
    comp_subset = df_copy[df_copy['Medicine'] == comparison]
    
    # Skip if either subset is empty
    if len(ref_subset) == 0 or len(comp_subset) == 0:
        return None
    
    # Get outcome counts - ensure they're numeric
    try:
        ref_positive = ref_subset[outcome_col].astype(int).sum()
        ref_negative = len(ref_subset) - ref_positive
        
        comp_positive = comp_subset[outcome_col].astype(int).sum()
        comp_negative = len(comp_subset) - comp_positive
        
        # Create contingency table
        table = [[ref_positive, ref_negative], [comp_positive, comp_negative]]
        
        # Use Fisher's exact test for small cell counts
        p_value = fisher_exact(table)[1]
        
        # Calculate odds ratio and CI
        or_value, ci_lower, ci_upper = calculate_odds_ratio(
            ref_positive, ref_negative, comp_positive, comp_negative
        )
        
        # Create results dictionary
        results = {
            'Reference': reference,
            'Comparison': comparison,
            'Ref_N': len(ref_subset),
            'Ref_Positive': ref_positive,
            'Ref_Positive_Pct': round((ref_positive / len(ref_subset)) * 100, 1) if len(ref_subset) > 0 else 0,
            'Comp_N': len(comp_subset),
            'Comp_Positive': comp_positive,
            'Comp_Positive_Pct': round((comp_positive / len(comp_subset)) * 100, 1) if len(comp_subset) > 0 else 0,
            'OR': or_value,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper,
            'P_Value': p_value
        }
        
        return results
    except Exception as e:
        print(f"Error in pairwise comparison {reference} vs {comparison}: {str(e)}")
        return None
        
# --------------------------------
# Table 3: Pairwise Comparisons
# --------------------------------
def create_pairwise_comparison_table(dfw, dfa, dfr):
    """Creates a table of pairwise comparisons of P. niruri vs. standard treatments."""
    comparisons = []
    
    # Define comparison treatments by platform based on sufficient high-quality reviews
    comparisons_by_platform = {
        'Amazon': ['Potassium citrate', 'Rowatinex'],
        'Reddit': ['Flomax', 'Potassium citrate', 'Allopurinol']
    }
    
    # Define reference treatment
    reference = 'Chanca piedra'
    
    # Platforms
    platforms = {
        'Amazon': dfa,
        'Reddit': dfr
    }
    
    # Perform comparisons for effectiveness
    for platform_name, treatments in comparisons_by_platform.items():
        df = platforms[platform_name]
        for comparison_treatment in treatments:
            try:
                effectiveness_results = pairwise_comparison(
                    df, reference, comparison_treatment, 
                    'Helps overall with kidney stones', high_quality=True
                )
                
                side_effects_results = pairwise_comparison(
                    df, reference, comparison_treatment, 
                    'Side effects mentioned', high_quality=True
                )
                
                if effectiveness_results and side_effects_results:
                    results = {
                        'Platform': platform_name,
                        'Comparison': comparison_treatment,
                        'P. niruri N': effectiveness_results['Ref_N'],
                        'Comparison N': effectiveness_results['Comp_N'],
                        'P. niruri Effectiveness %': effectiveness_results['Ref_Positive_Pct'],
                        'Comparison Effectiveness %': effectiveness_results['Comp_Positive_Pct'],
                        'Effectiveness OR': effectiveness_results['OR'],
                        'Effectiveness CI Lower': effectiveness_results['CI_Lower'],
                        'Effectiveness CI Upper': effectiveness_results['CI_Upper'],
                        'Effectiveness P-value': effectiveness_results['P_Value'],
                        'P. niruri Side Effects %': side_effects_results['Ref_Positive_Pct'],
                        'Comparison Side Effects %': side_effects_results['Comp_Positive_Pct'],
                        'Side Effects OR': side_effects_results['OR'],
                        'Side Effects CI Lower': side_effects_results['CI_Lower'],
                        'Side Effects CI Upper': side_effects_results['CI_Upper'],
                        'Side Effects P-value': side_effects_results['P_Value']
                    }
                    comparisons.append(results)
            except Exception as e:
                print(f"Error comparing {reference} vs {comparison_treatment} on {platform_name}: {e}")
    
    # Create DataFrame from comparisons
    if comparisons:
        comparison_df = pd.DataFrame(comparisons)
        
        # Format the odds ratios and CIs
        comparison_df['Effectiveness OR (95% CI)'] = comparison_df.apply(
            lambda row: f"{row['Effectiveness OR']:.2f} ({row['Effectiveness CI Lower']:.2f}-{row['Effectiveness CI Upper']:.2f})",
            axis=1
        )
        
        comparison_df['Side Effects OR (95% CI)'] = comparison_df.apply(
            lambda row: f"{row['Side Effects OR']:.2f} ({row['Side Effects CI Lower']:.2f}-{row['Side Effects CI Upper']:.2f})",
            axis=1
        )
        
        # Add significance indicators
        comparison_df['Effectiveness Sig'] = comparison_df['Effectiveness P-value'].apply(
            lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        )
        
        comparison_df['Side Effects Sig'] = comparison_df['Side Effects P-value'].apply(
            lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        )
        
        # Select columns for final table
        final_columns = [
            'Platform', 'Comparison', 'P. niruri N', 'Comparison N', 
            'P. niruri Effectiveness %', 'Comparison Effectiveness %',
            'Effectiveness OR (95% CI)', 'Effectiveness Sig', 
            'P. niruri Side Effects %', 'Comparison Side Effects %',
            'Side Effects OR (95% CI)', 'Side Effects Sig'
        ]
        
        comparison_table = comparison_df[final_columns]
        
        # Save the table
        comparison_table.to_html(figfolder + 'table3_pairwise_comparisons.html')
        comparison_table.to_csv(figfolder + 'table3_pairwise_comparisons.csv')
        
        return comparison_table
    
    return pd.DataFrame()

# --------------------------------
# Table 5: Multivariable Analysis
# --------------------------------
def pairwise_treatment_or(
    df: pd.DataFrame,
    outcome: str,
    covariates: list,
    treatment_col: str,
    treatments: list
) -> pd.DataFrame:
    """
    For each treatment in `treatments`, re-fit a logistic model using that
    treatment as the reference, then extract ORs & 95% CIs comparing all
    other treatments to it.
    """
    rows = []
    for ref in treatments:
        # tell Patsy that 'ref' is the baseline
        df['_treat_cat'] = df[treatment_col].astype(
            pd.CategoricalDtype(categories=[ref] + [t for t in treatments if t != ref])
        )

        # build model formula
        terms = [f"C(_treat_cat, PatsyTreatment(reference='{ref}'))"] + covariates
        model = smf.logit(f"{outcome} ~ " + " + ".join(terms), data=df).fit(disp=False)

        # pull out each non-ref coefficient
        for cmp in treatments:
            if cmp == ref:
                continue
            term = f"C(_treat_cat, PatsyTreatment(reference='{ref}'))[T.{cmp}]"
            coef = model.params[term]
            se   = model.bse[term]
            or_  = np.exp(coef)
            ci_l = np.exp(coef - 1.96 * se)
            ci_h = np.exp(coef + 1.96 * se)
            p    = model.pvalues[term]
            rows.append({
                'reference':  ref,
                'compare_to': cmp,
                'OR':          or_,
                'CI_lower':    ci_l,
                'CI_upper':    ci_h,
                'pvalue':      p
            })
    return pd.DataFrame(rows)

def create_multivariable_analysis(
    df: pd.DataFrame,
    outcome: str = 'helped',
    treatment_col: str = 'Treatment',
    quality_col: str = 'High_Quality',
    long_term_col: str = 'Long_Term_User',
    platform_col: str = 'Platform',
    treatments: list = None,
    min_reviews: int = 20,
    min_high_quality: int = 10
):
    """
    1) Filters treatments by minimum total and high-quality review counts.
    2) Fits a primary logistic model with the first treatment as reference.
    3) Runs pairwise comparisons among remaining treatments.

    Returns:
      - main_model: the fitted statsmodels Logit result
      - pairwise_df: DataFrame of all pairwise ORs + CIs + p-values
    """
    # 1) pick your treatments (by volume & HQ)
    if treatments is None:
        total_counts = df[treatment_col].value_counts()
        hq_counts    = df[df[quality_col]][treatment_col].value_counts()
        treatments = [
            t for t in total_counts.index
            if total_counts[t] >= min_reviews and hq_counts.get(t, 0) >= min_high_quality
        ]

    # 2) choose the first as your primary reference
    ref = treatments[0]

    # 3) build & fit the main model
    df['_treat_cat'] = df[treatment_col].astype(
        pd.CategoricalDtype(categories=[ref] + [t for t in treatments if t != ref])
    )
    # force platform reference to the most common one
    plat_ref = df[platform_col].mode()[0]
    covars = [
        quality_col,
        long_term_col,
        f"C({platform_col}, PatsyTreatment(reference='{plat_ref}'))"
    ]
    formula = f"{outcome} ~ C(_treat_cat, PatsyTreatment(reference='{ref}')) + " + " + ".join(covars)
    main_model = smf.logit(formula, data=df).fit(disp=False)

    # 4) get every pairwise OR among chosen treatments
    pairwise_df = pairwise_treatment_or(
        df,
        outcome=outcome,
        covariates=covars,
        treatment_col=treatment_col,
        treatments=treatments
    )

    return main_model, pairwise_df

# --------------------------------
# Figure 1: Forest Plot of Effectiveness Odds Ratios
# --------------------------------
def create_effectiveness_forest_plot(dfw, dfa, dfr):
    """Creates a forest plot showing odds ratios for effectiveness compared to P. niruri."""
    # List to store comparison results
    comparisons = []
    
    # Define reference treatment
    reference = 'Chanca piedra'
    
    # Define platforms
    platforms = {
        'WebMD': dfw,
        'Amazon': dfa,
        'Reddit': dfr
    }
    
    # Get all treatments across platforms
    all_treatments = set()
    for df in [dfw, dfa, dfr]:
        all_treatments.update(df['Medicine'].unique())
    
    # Remove reference treatment and non-kidney stone treatments
    excluded_treatments = [reference, 'Ashwagandha', 'Melatonin']
    comparison_treatments = [t for t in all_treatments if t not in excluded_treatments]
    
    # Calculate odds ratios for each platform and treatment
    for platform_name, df in platforms.items():
        for treatment in comparison_treatments:
            try:
                # Make sure both treatments exist in this platform
                if reference in df['Medicine'].unique() and treatment in df['Medicine'].unique():
                    # Get sample sizes
                    ref_n = len(df[df['Medicine'] == reference])
                    comp_n = len(df[df['Medicine'] == treatment])
                    
                    if ref_n >= 5 and comp_n >= 5:  # Only include if enough samples
                        # Calculate effectiveness comparison
                        results = pairwise_comparison(
                            df, reference, treatment, 'Helps overall with kidney stones', high_quality=False
                        )
                        
                        if results:
                            # Add to comparisons
                            comparisons.append({
                                'Platform': platform_name,
                                'Treatment': treatment,
                                'OR': results['OR'],
                                'CI_Lower': results['CI_Lower'],
                                'CI_Upper': results['CI_Upper'],
                                'Ref_N': results['Ref_N'],
                                'Comp_N': results['Comp_N'],
                                'P_Value': results['P_Value']
                            })
            except Exception as e:
                print(f"Error in effectiveness forest plot, {platform_name}/{treatment}: {e}")
    
    # Create DataFrame
    if comparisons:
        forest_df = pd.DataFrame(comparisons)
        
        # Create the forest plot
        fig = go.Figure()
        
        # Add data points grouped by platform
        platforms_order = ['WebMD', 'Amazon', 'Reddit']
        colors = ['blue', 'red', 'green']
        
        for i, platform in enumerate(platforms_order):
            platform_data = forest_df[forest_df['Platform'] == platform]
            
            if len(platform_data) > 0:
                fig.add_trace(go.Scatter(
                    x=platform_data['OR'],
                    y=platform_data['Treatment'],
                    error_x=dict(
                        type='data',
                        symmetric=False,
                        array=platform_data['CI_Upper'] - platform_data['OR'],
                        arrayminus=platform_data['OR'] - platform_data['CI_Lower']
                    ),
                    mode='markers',
                    name=platform,
                    marker=dict(size=10, color=colors[i]),
                    hovertemplate=(
                        "Treatment: %{y}<br>" +
                        "Platform: " + platform + "<br>" +
                        "Odds Ratio: %{x:.2f}<br>" +
                        "95% CI: (%{error_x.arrayminus:.2f}, %{error_x.array:.2f})<br>" +
                        "Sample sizes: P. niruri: %{customdata[0]}, Comparison: %{customdata[1]}<br>" +
                        "p-value: %{customdata[2]:.4f}" +
                        "<extra></extra>"
                    ),
                    customdata=platform_data[['Ref_N', 'Comp_N', 'P_Value']].values
                ))
        
        # Add vertical line at OR=1 (no effect)
        fig.add_shape(
            type="line",
            x0=1, x1=1,
            y0=-1, y1=len(comparison_treatments) + 0.5,
            line=dict(color="gray", width=1, dash="dash")
        )
        
        # Update layout
        fig.update_layout(
            title="Effectiveness Odds Ratios (P. niruri as reference)",
            xaxis_title="Odds Ratio (log scale)",
            yaxis_title=None,
            template="plotly_white",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="center",
                x=0.5
            ),
            xaxis=dict(
                type="log",
                range=[-1, 2]  # log10 scale: 0.1 to 100
            ),
            width=800,
            height=500
        )
        
        # Save figure
        fig.write_html(figfolder + "figure1_effectiveness_forest_plot.html")
        fig.write_image(figfolder + "figure1_effectiveness_forest_plot.png")
        
        return fig
    
    return None

# --------------------------------
# Figure 2: Forest Plot of Side Effect Odds Ratios
# --------------------------------
def create_side_effects_forest_plot(dfw, dfa, dfr):
    """Creates an improved forest plot showing odds ratios for side effects compared to P. niruri."""
    # List to store comparison results
    comparisons = []
    
    # Define reference treatment
    reference = 'Chanca piedra'
    
    # Define platforms
    platforms = {
        'WebMD': dfw,
        'Amazon': dfa,
        'Reddit': dfr
    }
    
    # Get all treatments across platforms
    all_treatments = set()
    for df in [dfw, dfa, dfr]:
        all_treatments.update(df['Medicine'].unique())
    
    # Remove reference treatment and non-kidney stone treatments
    excluded_treatments = [reference, 'Ashwagandha', 'Melatonin']
    comparison_treatments = [t for t in all_treatments if t not in excluded_treatments]
    
    # Calculate odds ratios for each platform and treatment
    for platform_name, df in platforms.items():
        for treatment in comparison_treatments:
            try:
                # Make sure both treatments exist in this platform
                if reference in df['Medicine'].unique() and treatment in df['Medicine'].unique():
                    # Get sample sizes
                    ref_n = len(df[df['Medicine'] == reference])
                    comp_n = len(df[df['Medicine'] == treatment])
                    
                    # Reduced minimum sample size to ensure more data points are included
                    # Original was likely 5, reduce to 3 or even 1 to include more points
                    if ref_n >= 1 and comp_n >= 1:  # Include all comparisons with at least 1 review
                        # Calculate side effects comparison
                        results = pairwise_comparison(
                            df, reference, treatment, 'Side effects mentioned', high_quality=False
                        )
                        
                        if results:
                            # Add to comparisons
                            comparisons.append({
                                'Platform': platform_name,
                                'Treatment': treatment,
                                'OR': results['OR'],
                                'CI_Lower': results['CI_Lower'],
                                'CI_Upper': results['CI_Upper'],
                                'Ref_N': results['Ref_N'],
                                'Comp_N': results['Comp_N'],
                                'P_Value': results['P_Value'],
                                'Ref_Pct': results['Ref_Positive_Pct'],
                                'Comp_Pct': results['Comp_Positive_Pct']
                            })
            except Exception as e:
                print(f"Error in side effects forest plot, {platform_name}/{treatment}: {e}")
    
    # Create DataFrame
    if comparisons:
        forest_df = pd.DataFrame(comparisons)
        
        # Print the data for debugging
        # print("Forest plot data:")
        # for idx, row in forest_df.iterrows():
        #     print(f"{row['Platform']} - {row['Treatment']}: OR={row['OR']:.2f}, CI=({row['CI_Lower']:.2f}-{row['CI_Upper']:.2f}), P={row['P_Value']:.4f}")
        
        # Create the forest plot
        fig = go.Figure()
        
        # Add data points grouped by platform
        platforms_order = ['WebMD', 'Amazon', 'Reddit']
        colors = ['blue', 'red', 'green']
        
        for i, platform in enumerate(platforms_order):
            platform_data = forest_df[forest_df['Platform'] == platform]
            
            if len(platform_data) > 0:
                fig.add_trace(go.Scatter(
                    x=platform_data['OR'],
                    y=platform_data['Treatment'],
                    error_x=dict(
                        type='data',
                        symmetric=False,
                        array=platform_data['CI_Upper'] - platform_data['OR'],
                        arrayminus=platform_data['OR'] - platform_data['CI_Lower']
                    ),
                    mode='markers',
                    name=platform,
                    marker=dict(size=10, color=colors[i]),
                    hovertemplate=(
                        "Treatment: %{y}<br>" +
                        "Platform: " + platform + "<br>" +
                        "Odds Ratio: %{x:.2f}<br>" +
                        "95% CI: (%{error_x.arrayminus:.2f}, %{error_x.array:.2f})<br>" +
                        "Side Effects: P. niruri: %{customdata[0]}%, Comparison: %{customdata[1]}%<br>" +
                        "Sample sizes: P. niruri: %{customdata[2]}, Comparison: %{customdata[3]}<br>" +
                        "p-value: %{customdata[4]:.4f}" +
                        "<extra></extra>"
                    ),
                    customdata=platform_data[['Ref_Pct', 'Comp_Pct', 'Ref_N', 'Comp_N', 'P_Value']].values
                ))
        
        # Add vertical line at OR=1 (no effect)
        fig.add_shape(
            type="line",
            x0=1, x1=1,
            y0=-1, y1=len(comparison_treatments) + 0.5,
            line=dict(color="gray", width=1, dash="dash")
        )
        
        # Update layout with extended x-axis range to ensure all points are visible
        # Use a wider range for the log scale to capture very small values
        fig.update_layout(
            title="Side Effect Odds Ratios (P. niruri as reference)",
            xaxis_title="Odds Ratio (log scale)",
            yaxis_title=None,
            template="plotly_white",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="center",
                x=0.5
            ),
            xaxis=dict(
                type="log",
                range=[-2, 2]  # Extended range: 0.01 to 100 (instead of original 0.1 to 100)
            ),
            width=800,
            height=500
        )
        
        # Correct annotation explaining the interpretation
        fig.add_annotation(
            x=0.15,
            y=-0.15,
            text="OR < 1: P. niruri has fewer side effects than comparison treatment<br>OR > 1: P. niruri has more side effects than comparison treatment",
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=10)
        )
        
        # Save figure
        fig.write_html(figfolder + "figure2_side_effects_forest_plot.html")
        fig.write_image(figfolder + "figure2_side_effects_forest_plot.png")
        
        return fig
    
    return None

# --------------------------------
# Figure 3: Stacked Bar Chart of Effectiveness Categories
# --------------------------------
def create_effectiveness_stacked_bar(dfw, dfa, dfr):
    """Creates a stacked bar chart showing the distribution of effectiveness outcomes."""
    # Combine all datasets
    combined_df = pd.concat([
        dfw.assign(Platform='WebMD'),
        dfa.assign(Platform='Amazon'),
        dfr.assign(Platform='Reddit')
    ])
    
    # Ensure columns are numeric
    combined_df['Helps overall with kidney stones'] = combined_df['Helps overall with kidney stones'].astype(int)
    
    # Get treatments with enough reviews
    treatment_counts = combined_df['Medicine'].value_counts()
    treatments = treatment_counts[treatment_counts >= 20].index.tolist()
    
    # Filter out non-kidney stone treatments
    excluded_treatments = ['Ashwagandha', 'Melatonin']
    treatments = [t for t in treatments if t not in excluded_treatments]
    
    # Prepare data for plotting
    plot_data = []
    
    for treatment in treatments:
        for platform in ['WebMD', 'Amazon', 'Reddit']:
            # Get subset
            subset = combined_df[(combined_df['Medicine'] == treatment) & 
                               (combined_df['Platform'] == platform)]
            
            # Skip if no reviews for this combination
            if len(subset) == 0:
                continue
            
            # Calculate counts
            helped_count = subset['Helps overall with kidney stones'].sum()
            
            # For Reddit data, many reviews might not explicitly state effectiveness
            # Count only explicit "not helped" (0 values)
            not_helped_count = sum(subset['Helps overall with kidney stones'] == 0)
            
            no_info_count = len(subset) - helped_count - not_helped_count
            
            # Calculate percentages
            total = len(subset)
            helped_pct = (helped_count / total) * 100
            not_helped_pct = (not_helped_count / total) * 100
            no_info_pct = (no_info_count / total) * 100
            
            # Add data points
            plot_data.append({
                'Treatment': treatment,
                'Platform': platform,
                'Category': 'Helped',
                'Percentage': helped_pct,
                'Count': helped_count,
                'Total': total
            })
            
            plot_data.append({
                'Treatment': treatment,
                'Platform': platform,
                'Category': 'Not Helped',
                'Percentage': not_helped_pct,
                'Count': not_helped_count,
                'Total': total
            })
            
            plot_data.append({
                'Treatment': treatment,
                'Platform': platform,
                'Category': 'No Information',
                'Percentage': no_info_pct,
                'Count': no_info_count,
                'Total': total
            })
    
    # Create DataFrame
    plot_df = pd.DataFrame(plot_data)
    
    # Define category order and colors
    category_order = ['Helped', 'Not Helped', 'No Information']
    category_colors = {'Helped': 'green', 'Not Helped': 'red', 'No Information': 'gray'}
    
    # Create stacked bar chart
    fig = px.bar(
        plot_df,
        x='Treatment',
        y='Percentage',
        color='Category',
        facet_col='Platform',
        category_orders={'Category': category_order},
        color_discrete_map=category_colors,
        barmode='stack',
        width=1000,
        height=500
    )
    
    # Update layout
    fig.update_layout(
        title='Effectiveness Categories by Treatment and Platform',
        xaxis_title=None,
        yaxis_title='Percentage of Reviews',
        legend_title=None,
        template='plotly_white',
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5
        )
    )
    
    # Update hover template
    fig.update_traces(
        hovertemplate=(
            "Treatment: %{x}<br>" +
            "Category: %{customdata[0]}<br>" +
            "Percentage: %{y:.1f}%<br>" +
            "Count: %{customdata[1]} / %{customdata[2]}" +
            "<extra></extra>"
        ),
        customdata=plot_df[['Category', 'Count', 'Total']].values
    )
    
    # Save figure
    fig.write_html(figfolder + "figure3_effectiveness_stacked_bar.html")
    fig.write_image(figfolder + "figure3_effectiveness_stacked_bar.png")
    
    return fig

# --------------------------------
# Figure 4: Effectiveness vs. Side Effects Scatterplot
# --------------------------------
def create_effectiveness_vs_side_effects_plot(dfw, dfa, dfr):
    """Creates a scatter plot showing effectiveness vs. side effects across treatments."""
    # Combine all datasets
    platforms = {
        'WebMD': dfw,
        'Amazon': dfa,
        'Reddit': dfr
    }
    
    # Prepare data for plotting
    plot_data = []
    
    for platform_name, df in platforms.items():
        # Get treatments with at least 5 reviews
        treatment_counts = df['Medicine'].value_counts()
        treatments = treatment_counts[treatment_counts >= 5].index.tolist()
        
        # Filter out non-kidney stone treatments
        excluded_treatments = ['Ashwagandha', 'Melatonin']
        treatments = [t for t in treatments if t not in excluded_treatments]
        
        for treatment in treatments:
            try:
                # Get subset
                treatment_df = df[df['Medicine'] == treatment].copy()
                
                # Ensure columns are numeric
                treatment_df['Helps overall with kidney stones'] = treatment_df['Helps overall with kidney stones'].astype(int)
                treatment_df['Side effects mentioned'] = treatment_df['Side effects mentioned'].astype(int)
                
                # Calculate effectiveness and side effects rates
                helped_count = treatment_df['Helps overall with kidney stones'].sum()
                side_effects_count = treatment_df['Side effects mentioned'].sum()
                total = len(treatment_df)
                
                helped_pct = (helped_count / total) * 100
                side_effects_pct = (side_effects_count / total) * 100
                
                # Add data point
                plot_data.append({
                    'Treatment': treatment,
                    'Platform': platform_name,
                    'Effectiveness': helped_pct,
                    'Side Effects': side_effects_pct,
                    'Count': total
                })
            except Exception as e:
                print(f"Error processing {treatment} on {platform_name} for scatter plot: {e}")
    
    # Create DataFrame
    plot_df = pd.DataFrame(plot_data)
    
    # Create scatter plot
    fig = px.scatter(
        plot_df,
        x='Effectiveness',
        y='Side Effects',
        color='Platform',
        size='Count',
        size_max=50,
        hover_name='Treatment',
        text='Treatment',
        width=800,
        height=600
    )
    
    # Add quadrant dividing lines
    fig.add_shape(
        type="line",
        x0=50, x1=50,
        y0=0, y1=100,
        line=dict(color="gray", width=1, dash="dash")
    )
    
    fig.add_shape(
        type="line",
        x0=0, x1=100,
        y0=20, y1=20,
        line=dict(color="gray", width=1, dash="dash")
    )
    
    # Add quadrant labels
    quadrants = [
        {"x": 25, "y": 10, "text": "Low effectiveness<br>Low side effects"},
        {"x": 75, "y": 10, "text": "High effectiveness<br>Low side effects"},
        {"x": 25, "y": 60, "text": "Low effectiveness<br>High side effects"},
        {"x": 75, "y": 60, "text": "High effectiveness<br>High side effects"}
    ]
    
    for quadrant in quadrants:
        fig.add_annotation(
            x=quadrant["x"],
            y=quadrant["y"],
            text=quadrant["text"],
            showarrow=False,
            font=dict(size=10)
        )
    
    # Update layout
    fig.update_layout(
        title='Effectiveness vs. Side Effects by Treatment',
        xaxis_title='Effectiveness (%)',
        yaxis_title='Side Effects (%)',
        template='plotly_white',
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5
        ),
        xaxis=dict(range=[0, 100]),
        yaxis=dict(range=[0, 100])
    )
    
    # Update hover template
    fig.update_traces(
        hovertemplate=(
            "Treatment: %{hovertext}<br>" +
            "Platform: %{marker.color}<br>" +
            "Effectiveness: %{x:.1f}%<br>" +
            "Side Effects: %{y:.1f}%<br>" +
            "Sample Size: %{marker.size}" +
            "<extra></extra>"
        )
    )
    
    # Save figure
    fig.write_html(figfolder + "figure4_effectiveness_vs_side_effects.html")
    fig.write_image(figfolder + "figure4_effectiveness_vs_side_effects.png")
    
    return fig

# --------------------------------
# Figure 5: Sensitivity Analysis Comparing All vs. High-Quality Reviews
# --------------------------------
def create_sensitivity_analysis_plot(dfw, dfa, dfr):
    """Creates a grouped bar chart comparing results from all reviews vs. high-quality reviews."""
    # Define treatments with sufficient high-quality reviews
    treatments_with_hq = []
    
    # Combined dataset
    combined_df = pd.concat([
        dfw.assign(Platform='WebMD'),
        dfa.assign(Platform='Amazon'),
        dfr.assign(Platform='Reddit')
    ])
    
    # Ensure columns are numeric
    combined_df['Helps overall with kidney stones'] = combined_df['Helps overall with kidney stones'].astype(int)
    combined_df['Side effects mentioned'] = combined_df['Side effects mentioned'].astype(int)
    combined_df['Super high quality'] = combined_df['Super high quality'].astype(int)
    
    # Find treatments with at least 10 high-quality reviews
    hq_counts = combined_df[combined_df['Super high quality'] == 1]['Medicine'].value_counts()
    treatments_with_hq = hq_counts[hq_counts >= 10].index.tolist()
    
    # Filter out non-kidney stone treatments
    excluded_treatments = ['Ashwagandha', 'Melatonin']
    treatments_with_hq = [t for t in treatments_with_hq if t not in excluded_treatments]
    
    # Prepare data for plotting
    plot_data = []
    
    for treatment in treatments_with_hq:
        try:
            # Get subsets
            all_reviews = combined_df[combined_df['Medicine'] == treatment]
            hq_reviews = all_reviews[all_reviews['Super high quality'] == 1]
            
            # Calculate effectiveness rates
            all_helped_pct = (all_reviews['Helps overall with kidney stones'].sum() / len(all_reviews)) * 100
            hq_helped_pct = (hq_reviews['Helps overall with kidney stones'].sum() / len(hq_reviews)) * 100
            
            # Calculate side effect rates
            all_side_effects_pct = (all_reviews['Side effects mentioned'].sum() / len(all_reviews)) * 100
            hq_side_effects_pct = (hq_reviews['Side effects mentioned'].sum() / len(hq_reviews)) * 100
            
            # Add data points for effectiveness
            plot_data.append({
                'Treatment': treatment,
                'Review Type': 'All Reviews',
                'Metric': 'Effectiveness',
                'Percentage': all_helped_pct,
                'Count': all_reviews['Helps overall with kidney stones'].sum(),
                'Total': len(all_reviews)
            })
            
            plot_data.append({
                'Treatment': treatment,
                'Review Type': 'High-Quality Only',
                'Metric': 'Effectiveness',
                'Percentage': hq_helped_pct,
                'Count': hq_reviews['Helps overall with kidney stones'].sum(),
                'Total': len(hq_reviews)
            })
            
            # Add data points for side effects
            plot_data.append({
                'Treatment': treatment,
                'Review Type': 'All Reviews',
                'Metric': 'Side Effects',
                'Percentage': all_side_effects_pct,
                'Count': all_reviews['Side effects mentioned'].sum(),
                'Total': len(all_reviews)
            })
            
            plot_data.append({
                'Treatment': treatment,
                'Review Type': 'High-Quality Only',
                'Metric': 'Side Effects',
                'Percentage': hq_side_effects_pct,
                'Count': hq_reviews['Side effects mentioned'].sum(),
                'Total': len(hq_reviews)
            })
        except Exception as e:
            print(f"Error processing {treatment} for sensitivity analysis: {e}")
    
    # Create DataFrame
    plot_df = pd.DataFrame(plot_data)
    
    # Create grouped bar chart
    fig = px.bar(
        plot_df,
        x='Treatment',
        y='Percentage',
        color='Review Type',
        facet_row='Metric',
        barmode='group',
        width=800,
        height=600
    )
    
    # Update layout
    fig.update_layout(
        title='Sensitivity Analysis: All Reviews vs. High-Quality Reviews',
        xaxis_title=None,
        yaxis_title='Percentage',
        template='plotly_white',
        legend_title=None,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5
        )
    )
    
    # Update hover template
    fig.update_traces(
        hovertemplate=(
            "Treatment: %{x}<br>" +
            "Review Type: %{marker.color}<br>" +
            "Percentage: %{y:.1f}%<br>" +
            "Count: %{customdata[0]} / %{customdata[1]}" +
            "<extra></extra>"
        ),
        customdata=plot_df[['Count', 'Total']].values
    )
    
    # Save figure
    fig.write_html(figfolder + "figure5_sensitivity_analysis.html")
    fig.write_image(figfolder + "figure5_sensitivity_analysis.png")
    
    return fig

In [33]:
# --------------------------------
# Main function to generate all tables and figures
# --------------------------------
def generate_all_tables_and_figures():
    """Generate all tables and figures for the study."""
    try:
        # Load data for each platform
        dfw, dfa, dfr = load_data()
        if dfw is None or dfa is None or dfr is None:
            print("Error loading data. Aborting analysis.")
            return

        # 0) Combine into a single DataFrame
        combined_df = pd.concat([
            dfw.assign(Platform='WebMD'),
            dfa.assign(Platform='Amazon'),
            dfr.assign(Platform='Reddit')
        ], ignore_index=True)
        combined_df = combined_df.rename(columns={
            'Helps overall with kidney stones': 'helped',
            'Super high quality':               'High_Quality',
            'Says that has suffered from condition for a long time (>1 year)': 'Long_Term_User',
            'Medicine':                          'Treatment'
        })


        print("Generating tables and figures...")

        # 1) Table 1: Review Summary
        print("  Generating Table 1: Review Summary")
        table1 = create_review_summary_table(dfw, dfa, dfr)

        # 2) Table 2: Effectiveness and Side Effects
        print("  Generating Table 2: Effectiveness and Side Effects")
        table2 = create_effectiveness_table(dfw, dfa, dfr)

        # 3) Table 3: Pairwise Comparisons
        print("  Generating Table 3: Pairwise Comparisons")
        table3 = create_pairwise_comparison_table(dfw, dfa, dfr)

        # 4) Table 5: Multivariable Analysis
        print("  Generating Table 5: Multivariable Analysis")
        main_model, table5 = create_multivariable_analysis(
            combined_df,
            outcome='helped',
            treatment_col='Treatment',
            quality_col='High_Quality',
            long_term_col='Long_Term_User',
            platform_col='Platform',
            treatments=[ 
                'Potassium citrate',
                'Hydrochlorothiazide',
                'Allopurinol',
                'Flomax',
                'Chanca piedra'
            ],
            min_reviews=20,
            min_high_quality=10
        )

        # optionally save or export table5:
        table5.to_csv(figfolder + 'table5_multivariable_analysis.csv', index=False)
        table5.to_html(figfolder + 'table5_multivariable_analysis.html', index=False)

        # 5) Figures
        print("  Generating Figure 1: Effectiveness Forest Plot")
        fig1 = create_effectiveness_forest_plot(dfw, dfa, dfr)

        print("  Generating Figure 2: Side Effects Forest Plot")
        fig2 = create_side_effects_forest_plot(dfw, dfa, dfr)

        print("  Generating Figure 3: Effectiveness Stacked Bar")
        fig3 = create_effectiveness_stacked_bar(dfw, dfa, dfr)

        print("  Generating Figure 4: Effectiveness vs Side Effects")
        fig4 = create_effectiveness_vs_side_effects_plot(dfw, dfa, dfr)

        print("  Generating Figure 5: Sensitivity Analysis")
        fig5 = create_sensitivity_analysis_plot(dfw, dfa, dfr)

        print("All tables and figures generated successfully!")

    except Exception as e:
        print(f"Error generating tables and figures: {e}")


generate_all_tables_and_figures()

Successfully loaded WebMD: 1567 rows, Amazon: 1456 rows, Reddit: 2308 rows
Generating tables and figures...
  Generating Table 1: Review Summary
  Generating Table 2: Effectiveness and Side Effects
  Generating Table 3: Pairwise Comparisons
  Generating Table 5: Multivariable Analysis
  Generating Figure 1: Effectiveness Forest Plot
  Generating Figure 2: Side Effects Forest Plot
  Generating Figure 3: Effectiveness Stacked Bar
  Generating Figure 4: Effectiveness vs Side Effects
  Generating Figure 5: Sensitivity Analysis
All tables and figures generated successfully!
